In [29]:
import math
from typing import Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd

data = pd.read_csv('./data/binanceusdm_swap_BTC-USDT_1h.csv', index_col='datetime', parse_dates=True)
data = data[data.index > "2025-04-30 18:00:00+08:00"].reset_index(drop=True)
predictions = pd.read_csv('./data/prediction3.csv')
predictions = predictions[predictions['t0'] > "2025-04-30 18:00:00+08:00"].reset_index(drop=True)
labels = pd.read_csv('./data/BTC-USDT_1h_atr_lookback36_label.csv')


In [32]:
long_condition = (predictions['side'] == 1) & (predictions['pred_vote'] ==1)
long_only = predictions[long_condition].dropna().reset_index(drop=True)

short_condition = (predictions['side'] == -1) & (predictions['pred_vote'] == 1)
short_only = predictions[short_condition].dropna().reset_index(drop=True)

print(f"long win ratio : {long_only['label'].value_counts()}")
print(f"short win ratio : {short_only['label'].value_counts()}")

long win ratio : label
1.0    14
0.0    12
Name: count, dtype: int64
short win ratio : label
1.0    28
Name: count, dtype: int64


In [ ]:
long_entry = pd.Series([False] * len(data), index=data.index)
long_exit = pd.Series([False] * len(data), index=data.index)
short_entry = pd.Series([False] * len(data), index=data.index)
short_exit = pd.Series([False] * len(data), index=data.index)
close = data['close']

for _,event in long_only.iterrows():
    
    t0 = event.loc['t0']
    t1 = event.loc['t1']
    long_entry.at[t0] = True
    long_exit.at[t1] = True

for _,event in short_only.iterrows():
    t0 = event.loc['t0']
    t1 = event.loc['t1']
    short_entry.at[t0] = True
    short_exit.at[t1] = True


In [ ]:
import vectorbt as vbt

long_pf = vbt.Portfolio.from_signals(
    close,
    long_entry,
    long_exit,
    init_cash=10000,
    fees=0.001,
    slippage=0.001,
    size_type= "value",
    size=1000,
)

long_out = {
    "portfolio": long_pf,
    "stats": long_pf.stats(),
    "value_raw": long_pf.value(),
    "orders": getattr(long_pf.orders, "records_readable", long_pf.orders),
    "trades": getattr(long_pf.trades, "records_readable", long_pf.trades),
}

print(long_out['stats'])

short_pf = vbt.Portfolio.from_signals(
    close,
    short_entry,
    short_exit,
    init_cash=10000,
    fees=0.001,
    slippage=0.001,
    size_type= "value",
    size=1000,
)

short_out = {
    "portfolio": short_pf,
    "stats": short_pf.stats(),
    "value_raw": short_pf.value(),
    "orders": getattr(short_pf.orders, "records_readable", short_pf.orders),
    "trades": getattr(short_pf.trades, "records_readable", short_pf.trades),
}
print(short_out['stats'])



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/louisexpc/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/louisexpc/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/louisexpc/.local/lib/python3.10

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import